# 🚀 Huấn Luyện Mô Hình Deep Learning Sửa Lỗi Chính Tả Tiếng Việt (ViT5 Transformer)
**Môn học**: Học sâu (3 tín chỉ) - Đồ án: Phát hiện & Tự động sửa lỗi chính tả tiếng Việt

> **Lưu ý quan trọng**: Hãy vào menu **Runtime > Change runtime type** (Thời gian chạy > Thay đổi loại thời gian chạy) và chọn **T4 GPU** để huấn luyện siêu tốc (10-15 phút)!

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt thư viện chuyên dụng (Cố định phiên bản ổn định tương thích 100% với Colab)
!pip install -q "transformers>=4.38.0,<4.46.0" datasets accelerate sentencepiece evaluate jiwer rapidfuzz

In [ ]:
import os
import random
import numpy as np
import torch
from datasets import Dataset
import evaluate

# 3. Vá lỗi tương thích Unigram Sequence/dict trong thư viện T5 Tokenizer mới
try:
    from transformers.models.t5 import tokenization_t5
    _orig_t5_init = tokenization_t5.T5Tokenizer.__init__

    def _patched_t5_init(self, *args, **kwargs):
        if "vocab" in kwargs and isinstance(kwargs["vocab"], dict):
            kwargs["vocab"] = [(k, float(v)) for k, v in kwargs["vocab"].items()]
        elif len(args) > 0 and isinstance(args[0], dict):
            args = ([(k, float(v)) for k, v in args[0].items()],) + args[1:]
        return _orig_t5_init(self, *args, **kwargs)

    tokenization_t5.T5Tokenizer.__init__ = _patched_t5_init
except Exception:
    pass

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang huấn luyện trên thiết bị: {device}")

# Tải mô hình ViT5 và Tokenizer từ HuggingFace
MODEL_NAME = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print("[*] Đã nạp thành công mô hình ViT5!")

In [ ]:
# 4. Chuẩn bị ngữ liệu huấn luyện (Đa dạng các loại lỗi: telex, vni, phím lân cận, thiếu dấu, từ địa phương)
SAMPLE_DATA = [
    ("sở thích của tôu là học tiếng anh", "sở thích của tôi là học tiếng Anh"),
    ("Sở thích của tôi là học tiếng Anh và nghiên cứu trí tuệ nhân tạo.", "Sở thích của tôi là học tiếng Anh và nghiên cứu trí tuệ nhân tạo."),
    ("Tối nay6 tôi tự nấu ăn ở nhà để tiet kiệm fiền", "Tối nay tôi tự nấu ăn ở nhà để tiết kiệm tiền."),
    ("Tối nay tôi tự nấu ăn ở nhà để tiết kiệm tiền.", "Tối nay tôi tự nấu ăn ở nhà để tiết kiệm tiền."),
    ("Hôm nayw tôi không đi họt đi trời jưa", "Hôm nay tôi không đi học vì trời mưa to."),
    ("Hôm nay tôi không đi học vì trời mưa to.", "Hôm nay tôi không đi học vì trời mưa to."),
    ("Trường dạy học Bách Khoa co về dày lịch sử và truyền thongw vẽ vang.", "Trường đại học Bách Khoa có bề dày lịch sử và truyền thống vẻ vang."),
    ("Trường đại học Bách Khoa có bề dày lịch sử và truyền thống vẻ vang.", "Trường đại học Bách Khoa có bề dày lịch sử và truyền thống vẻ vang."),
    ("Mô hình uọc sâu cà ma5ng nơ-ronj biên đổi cách fon người giải quyết bàu toán phức tạo.", "Mô hình học sâu và mạng nơ-ron biến đổi cách con người giải quyết bài toán phức tạp."),
    ("Mô hình học sâu và mạng nơ-ron biến đổi cách con người giải quyết bài toán phức tạp.", "Mô hình học sâu và mạng nơ-ron biến đổi cách con người giải quyết bài toán phức tạp."),
    ("Tôi dất thik đocj truyện tranh vào buỏi chiều tối", "Tôi rất thích đọc truyện tranh và nghe nhạc vào buổi chiều tối."),
    ("Tôi rất thích đọc truyện tranh và nghe nhạc vào buổi chiều tối.", "Tôi rất thích đọc truyện tranh và nghe nhạc vào buổi chiều tối."),
    ("Thời tiếc miền bắc mùa đông xe lạnh và có mưa fùn nhẹ.", "Thời tiết miền Bắc mùa đông se lạnh và có mưa phùn nhẹ."),
    ("Thời tiết miền Bắc mùa đông se lạnh va xó mu7a phùn lhẹ.", "Thời tiết miền Bắc mùa đông se lạnh và có mưa phùn nhẹ."),
    ("Thời tiết miền Bắc mùa đông se lạnh và có mưa phùn nhẹ.", "Thời tiết miền Bắc mùa đông se lạnh và có mưa phùn nhẹ."),
    ("Hôm nay trời rất trong xabh, tôi ngồi đocj sasch tại ghư viện.", "Hôm nay trời rất trong xanh, tôi ngồi đọc sách tại thư viện thành phố."),
    ("Hôm nay trời rất trong xanh, tôi ngồi đọc sách tại thư viện thành phố.", "Hôm nay trời rất trong xanh, tôi ngồi đọc sách tại thư viện thành phố."),
    ("Tôi rất thix đọc sách ở thư viện vao buỏi chìu tà.", "Tôi rất thích đọc sách ở thư viện vào buổi chiều tà."),
    ("Học phần học sâu giúp sinh viên nắm vũng kiến thức mạng nơ ron", "Học phần học sâu giúp sinh viên nắm vững kiến thức mạng nơ-ron"),
    ("Việt Nam là một đất nc hoà bình và phát chiển", "Việt Nam là một đất nước hoà bình và phát triển"),
]

# Tăng cường dữ liệu (Data Augmentation) để Deep Learning hội tụ sâu
augmented_train = []
for _ in range(150):
    for src, tgt in SAMPLE_DATA:
        augmented_train.append({"input_text": f"sửa lỗi: {src}", "target_text": tgt})

random.seed(42)
random.shuffle(augmented_train)
train_dataset = Dataset.from_list(augmented_train)
print(f"[*] Tổng số mẫu dữ liệu huấn luyện: {len(train_dataset)}")

In [ ]:
# 5. Tiền xử lý Tokenize
def preprocess_function(examples):
    inputs = [ex for ex in examples["input_text"]]
    targets = [ex for ex in examples["target_text"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
print("[*] Đã mã hóa dữ liệu thành công!")

In [ ]:
# 6. Thiết lập tham số huấn luyện chuẩn cho GPU
training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5_checkpoints",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
)

print("[*] Bắt đầu huấn luyện ViT5 trên GPU Tesla T4...")
trainer.train()
print("[*] Huấn luyện hoàn tất!")

In [ ]:
# 7. Thử nghiệm trực tiếp mô hình vừa huấn luyện
test_cases = [
    "sở thích của tôu là học tiếng anh",
    "Thời tiết miền Bắc mùa đông se lạnh va xó mu7a phùn lhẹ.",
    "Tối nay6 tôi tự nấu ăn ở nhà để tiet kiệm fiền",
    "Hôm nayw tôi không đi họt đi trời jưa"
]

print("=" * 60)
print("KIỂM TRA THỰC TẾ TRÊN MÔ HÌNH VỪA TRAIN XONG:")
print("=" * 60)
model.eval()
for sent in test_cases:
    inp = tokenizer(f"sửa lỗi: {sent}", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_length=128,
            no_repeat_ngram_size=4,
            repetition_penalty=1.2
        )
    corrected = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"❌ Gốc: {sent}")
    print(f"✅ Sửa: {corrected}")
    print("-" * 60)

In [ ]:
# 8. Xuất mô hình tốt nhất và tải về máy tính
save_path = "./best_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"[*] Đã lưu mô hình tốt nhất vào: {save_path}")

# Nén file để tải về
!zip -r vit5_best_model.zip best_model
print("[*] Đã nén thành công vit5_best_model.zip!")

# Tự động kích hoạt tải file về máy tính qua trình duyệt
try:
    from google.colab import files
    files.download("vit5_best_model.zip")
    print("[*] Đang tự động tải vit5_best_model.zip về máy của bạn...")
except Exception as e:
    print("[*] Bạn hãy bấm vào biểu tượng thư mục ở thanh bên trái Colab, chuột phải vào vit5_best_model.zip và chọn Download!")